# Mappa ambiti

Notebook dedicato agli ambiti territoriali delle reti. L'unità di analisi non è il singolo soggetto ma il file/rete.

**Nota metodologica**: la governance totale è l’aggregazione di coordinamento, gestione e monitoraggio. `Governance generica` indica partecipazione alla governance senza dettaglio della componente specifica. `Ente capofila` è un ruolo autonomo, distinto dalla governance totale.


In [43]:

from pathlib import Path
import json
import re
from collections import Counter
import pandas as pd
import folium
from folium.plugins import MarkerCluster
from IPython.display import display

INPUT_JSON = Path(r"output\data\step_3.0\3.0_risultati_enriched_merged_geo.json")
GEOJSON_REGIONI = Path(r"geo-json\limits_IT_regions.geojson")
GEOJSON_PROVINCE = Path(r"geo-json\limits_IT_provinces.geojson")
OUTPUT_MAPPA = Path(r"output\reports\step_3.4.1_html\3.4.1_mappa_ambiti.html")
OUTPUT_DIR_TABLE = Path(r"output\reports\step_3.4.1_csv")





In [44]:

def clean_text(x):
    if x is None:
        return ""
    return re.sub(r"\s+", " ", str(x).strip())

def clean_title(x):
    x = clean_text(x)
    return x.title() if x else ""

def to_float_or_none(x):
    try:
        return float(x)
    except Exception:
        return None

def iter_items(obj):
    if isinstance(obj, dict):
        if "soggetti" in obj or "entities" in obj:
            yield obj
        for v in obj.values():
            yield from iter_items(v)
    elif isinstance(obj, list):
        for item in obj:
            yield from iter_items(item)

def iter_entities(item):
    if not isinstance(item, dict):
        return []
    if isinstance(item.get("soggetti"), list):
        return [x for x in item["soggetti"] if isinstance(x, dict)]
    if isinstance(item.get("entities"), list):
        return [x for x in item["entities"] if isinstance(x, dict)]
    return []

def first_non_empty(*values):
    for v in values:
        v = clean_text(v)
        if v:
            return v
    return ""

def mode_or_empty(values):
    vals = [v for v in values if clean_text(v)]
    if not vals:
        return ""
    return Counter(vals).most_common(1)[0][0]

def detect_ambito_label(item, entities):
    explicit = first_non_empty(item.get("ambito_territoriale"), item.get("ambito"), item.get("livello_territoriale"), item.get("territorio"))
    if explicit:
        return explicit
    levels = [clean_text(e.get("livello_territoriale")).lower() for e in entities if clean_text(e.get("livello_territoriale"))]
    if not levels:
        return "non definito"
    if "regione" in levels:
        return "regionale"
    if "provincia" in levels:
        return "provinciale"
    if "comune" in levels:
        return "comunale"
    return mode_or_empty(levels) or "non definito"


In [45]:

with open(INPUT_JSON, "r", encoding="utf-8") as f:
    merged = json.load(f)
geo_regioni = None
geo_province = None
if GEOJSON_REGIONI.exists():
    with open(GEOJSON_REGIONI, "r", encoding="utf-8") as f:
        geo_regioni = json.load(f)
if GEOJSON_PROVINCE.exists():
    with open(GEOJSON_PROVINCE, "r", encoding="utf-8") as f:
        geo_province = json.load(f)


In [46]:

rows = []
for item in iter_items(merged):
    ents = iter_entities(item)
    if not ents:
        continue
    file_name = first_non_empty(item.get("file"), item.get("files"), item.get("titolo"), item.get("nome_file"))
    coords, comuni, province, regioni, livelli = [], [], [], [], []
    n_firmatari = n_proponenti = n_attori = n_governance = 0
    n_governance_generica = 0
    n_enti_capofila = 0
    enti_capofila = []
    for ent in ents:
        lat = to_float_or_none(ent.get("lat")); lon = to_float_or_none(ent.get("lon"))
        if lat is not None and lon is not None and -90 <= lat <= 90 and -180 <= lon <= 180:
            coords.append((lat, lon))
        comuni.append(clean_title(ent.get("comune") or ent.get("comune_matchato")))
        province.append(clean_title(ent.get("provincia")))
        regioni.append(clean_title(ent.get("regione")))
        livelli.append(clean_text(ent.get("livello_territoriale")).lower())
        n_firmatari += int(bool(ent.get("is_firmatario")))
        n_proponenti += int(bool(ent.get("is_proponente")))
        n_attori += int(bool(ent.get("is_attore")))
        n_governance += int(bool(ent.get("is_governance")))
        n_governance_generica += int(bool(ent.get("is_governance_generica")))
        if bool(ent.get("is_ente_capofila")):
            n_enti_capofila += 1
            nome_capofila = clean_text(ent.get("nome"))
            if nome_capofila:
                enti_capofila.append(nome_capofila)
    lat = sum(x for x, _ in coords) / len(coords) if coords else None
    lon = sum(y for _, y in coords) / len(coords) if coords else None
    rows.append({
        "file": file_name,
        "ambito_territoriale": detect_ambito_label(item, ents),
        "livello_prevalente": mode_or_empty(livelli),
        "comune_prevalente": mode_or_empty(comuni),
        "provincia_prevalente": mode_or_empty(province),
        "regione_prevalente": mode_or_empty(regioni),
        "lat": lat,
        "lon": lon,
        "n_soggetti": len(ents),
        "n_firmatari": n_firmatari,
        "n_proponenti": n_proponenti,
        "n_attori": n_attori,
        "n_governance": n_governance,
        "n_governance_generica": n_governance_generica,
        "n_enti_capofila": n_enti_capofila,
        "ente_capofila_principale": Counter(enti_capofila).most_common(1)[0][0] if enti_capofila else "",
    })

df_ambiti = pd.DataFrame(rows).copy()
if df_ambiti.empty:
    raise SystemExit("Nessun ambito costruito dal JSON enriched.")
for col in ["file", "ambito_territoriale", "livello_prevalente", "comune_prevalente", "provincia_prevalente", "regione_prevalente"]:
    df_ambiti[col] = df_ambiti[col].fillna("").astype(str).str.strip()
display(df_ambiti.head())


,file,ambito_territoriale,livello_prevalente,comune_prevalente,provincia_prevalente,regione_prevalente,lat,lon,n_soggetti,n_firmatari,n_proponenti,n_attori,n_governance,n_governance_generica,n_enti_capofila,ente_capofila_principale
0,01_2406_rta_01_240904110730_4166---protocolloc...,regionale,comune,Alba,Cuneo,Piemonte,44.689868,7.923946,9,5,1,5,1,0,2,Comune di Bra
1,01_2406_rta_01_240904120005_4166---protocollo.txt,regionale,comune,Biella,Biella,Piemonte,45.532730,8.038533,34,15,1,24,6,0,1,Consorzio Intercomunale Servizi Socio Assisten...
2,01_2406_rta_01_240904121349_4166---convenzione...,regionale,comune,Biella,Biella,Piemonte,45.531913,8.039566,29,12,0,15,1,0,0,
3,01_2406_rta_01_240904122512_4166---accordoSFD.txt,provinciale,comune,Biella,Biella,Piemonte,45.561168,8.059637,26,2,0,16,3,0,0,
4,01_2406_rta_01_240904150044_4166---Protocolloc...,provinciale,comune,Borgomanero,Novara,Piemonte,45.710010,8.525652,29,8,0,11,1,0,0,


In [47]:

def classify_ambito_bucket(x):
    xl = clean_text(x).lower()
    if "region" in xl:
        return "regionale"
    if "provinc" in xl or "metropolitan" in xl:
        return "provinciale"
    if "comunal" in xl or xl == "comune":
        return "comunale"
    return "altro / non definito"

df_ambiti["bucket_ambito"] = df_ambiti["ambito_territoriale"].apply(classify_ambito_bucket)
df_ambiti_valid = df_ambiti[df_ambiti["lat"].notna() & df_ambiti["lon"].notna()].copy()


In [48]:

def style_regioni(feature):
    return {"fillColor": "#00000000", "color": "#cc0000", "weight": 2, "fillOpacity": 0.0}

def style_province(feature):
    return {"fillColor": "#00000000", "color": "#666666", "weight": 1, "fillOpacity": 0.0}

def popup_ambito(row):
    return f"""
    <div style='font-size:13px;line-height:1.4;'>
      <b>File:</b> {row.get('file','')}<br>
      <b>Ambito territoriale:</b> {row.get('ambito_territoriale','')}<br>
      <b>Bucket:</b> {row.get('bucket_ambito','')}<br>
      <b>Livello prevalente:</b> {row.get('livello_prevalente','')}<br>
      <b>Comune prevalente:</b> {row.get('comune_prevalente','')}<br>
      <b>Provincia prevalente:</b> {row.get('provincia_prevalente','')}<br>
      <b>Regione prevalente:</b> {row.get('regione_prevalente','')}<br>
      <b>N. soggetti:</b> {row.get('n_soggetti','')}<br>
      <b>N. firmatari:</b> {row.get('n_firmatari','')}<br>
      <b>N. proponenti:</b> {row.get('n_proponenti','')}<br>
      <b>N. attori:</b> {row.get('n_attori','')}<br>
      <b>N. governance:</b> {row.get('n_governance','')}
    </div>
    """


In [49]:
def make_popup_ambito(row):
    html = f"""
    <div style="font-size: 13px; line-height: 1.4;">
        <b>File:</b> {row.get('file', '')}<br>
        <b>Regione:</b> {row.get('regione', '')}<br>
        <b>Provincia:</b> {row.get('provincia', '')}<br>
        <b>Comune:</b> {row.get('comune', '')}<br>
        <b>Livello prevalente:</b> {row.get('livello_prevalente', '')}<br>
        <hr>
        <b>Soggetti:</b> {row.get('n_soggetti', 0)}<br>
        <b>Firmatari:</b> {row.get('n_firmatari', 0)}<br>
        <b>Proponenti:</b> {row.get('n_proponenti', 0)}<br>
        <b>Attori:</b> {row.get('n_attori', 0)}<br>
        <b>Gestione:</b> {row.get('n_gestione', 0)}<br>
        <b>Monitoraggio:</b> {row.get('n_monitoraggio', 0)}<br>
        <b>Coordinamento:</b> {row.get('n_coordinamento', 0)}<br>
        <b>Governance totale:</b> {row.get('n_governance', 0)}<br>
        <b>Governance generica:</b> {row.get('n_governance_generica', 0)}<br>
        <b>Enti capofila:</b> {row.get('n_enti_capofila', 0)}<br>
        <b>Capofila principale:</b> {row.get('ente_capofila_principale', '')}<br>
    </div>
    """
    return html

In [50]:

if df_ambiti_valid.empty:
    raise SystemExit("Nessun ambito con coordinate valide.")
centro_lat = df_ambiti_valid["lat"].mean()
centro_lon = df_ambiti_valid["lon"].mean()
m = folium.Map(location=[centro_lat, centro_lon], zoom_start=6, tiles="CartoDB positron")
if geo_regioni is not None:
    folium.GeoJson(geo_regioni, name="Confini regioni", style_function=style_regioni).add_to(m)
if geo_province is not None:
    folium.GeoJson(geo_province, name="Confini province", style_function=style_province, show=False).add_to(m)
colors = {"comunale": "blue", "provinciale": "orange", "regionale": "red", "altro / non definito": "gray"}
for bucket, df_sub in df_ambiti_valid.groupby("bucket_ambito"):
    fg = folium.FeatureGroup(name=f"Ambiti {bucket}", show=True)
    cluster = MarkerCluster().add_to(fg)
    for _, row in df_sub.iterrows():
        folium.CircleMarker(location=[row["lat"], row["lon"]], radius=6, color=colors.get(bucket, "gray"), fill=True, fill_opacity=0.75, popup=folium.Popup(popup_ambito(row), max_width=480), tooltip=row.get("file", "")).add_to(cluster)
    fg.add_to(m)
folium.LayerControl(collapsed=False).add_to(m)
m


In [51]:
OUTPUT_MAPPA.parent.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR_TABLE.mkdir(parents=True, exist_ok=True)
m.save(OUTPUT_MAPPA)
df_ambiti.to_csv(OUTPUT_DIR_TABLE / "ambiti_reti.csv", index=False, encoding="utf-8-sig")
#(df_ambiti.groupby("bucket_ambito", dropna=False).size().reset_index(name="n_reti").sort_values("n_reti", ascending=False).to_csv(OUTPUT_DIR_TABLE / "tabella_ambiti_per_bucket.csv", index=False, encoding="utf-8-sig"))
print(f"Mappa salvata in: {OUTPUT_MAPPA}")
print(f"Tabelle salvate in: {OUTPUT_DIR_TABLE}")


Mappa salvata in: output\reports\step_3.4.1_html\3.4.1_mappa_ambiti.html
Tabelle salvate in: output\reports\step_3.4.1_csv


In [52]:
#display(df_ambiti.groupby("bucket_ambito", dropna=False).size().reset_index(name="n_reti").sort_values("n_reti", ascending=False))
#display(df_ambiti[["file", "ambito_territoriale", "bucket_ambito", "livello_prevalente", "comune_prevalente", "provincia_prevalente", "regione_prevalente", "n_soggetti", "n_firmatari", "n_proponenti", "n_attori", "n_governance"]].head(30))
